# Модуль 12. Разбор кейсов: Электронная энергия H2: RHF и DFT/PBE

**Модуль 12. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** Два атома H, расстояние 0,74 Å; стандартные параметры STO-3G заданы явно в h2_scf.py.

Проследить RHF-расчёт и сравнить HF/PBE при одинаковых геометрии и базисе.

**Workflow:** Геометрия/заряд/spin → RHF → кривая → PySCF HF/PBE → сходимость и ограничения.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab12_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%2012_quatium%20chemistry.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab12/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. Что входит в электронную энергию

Для H2 фиксируем расстояние, заряд 0, spin=Nα−Nβ=0 и базис STO-3G. Собственная реализация RHF показывает цикл «плотность → Фок → орбитали → новая плотность». Полная Born–Oppenheimer энергия включает отталкивание ядер. Сходимость решает численную задачу, но не доказывает адекватность приближения.

**Перед запуском:** Прочитайте ограничение реализации: только H2, закрытая оболочка, s-функции.

In [2]:
from h2_scf import rhf_h2
result=rhf_h2(.74)
print(result)
assert result['converged']
assert abs(result['energy_hartree']+1.116759307)<5e-5
display(pd.DataFrame(result['trace']))


{'distance_angstrom': 0.74, 'energy_hartree': -1.1167593073781579, 'converged': True, 'cycles': 2, 'trace': [{'cycle': 1, 'energy_hartree': -1.116759307378158, 'density_change': 5.438959822042073e-16}, {'cycle': 2, 'energy_hartree': -1.1167593073781579, 'density_change': 2.7194799110210365e-16}], 'method': 'RHF/STO-3G; restricted H2 only'}


,cycle,energy_hartree,density_change
0,1,-1.116759,5.438960e-16
1,2,-1.116759,2.719480e-16


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Что входит в электронную энергию — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Кривая и единицы

Сравниваем точки одной системы при неизменном методе. Разность Hartree переводим в kJ/mol указанным коэффициентом. Минимум сетки не заменяет оптимизацию геометрии и проверку частот. Растянутая связь обнаруживает ограничение однодетерминантного RHF; красивый SCF trace не устраняет статическую корреляцию.

**Перед запуском:** Посчитайте энергию и проверьте контрольное значение около −1,1167593 Hartree.

In [3]:
assert math.isclose(-.010*2625.5,-26.255,abs_tol=1e-9)
scan=pd.DataFrame([{'R_A':float(r),'E_hartree':rhf_h2(float(r))['energy_hartree']}
                   for r in np.linspace(.5,2.5,11)])
scan.to_csv(OUT/'h2_scan.csv',index=False)
plt.plot(scan.R_A,scan.E_hartree,'o-');plt.xlabel('H–H, angstrom');plt.ylabel('RHF/STO-3G E, Hartree')
plt.savefig(OUT/'h2_energy.png',dpi=150);plt.show()


/tmp/ipykernel_14794/151431294.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(OUT/'h2_energy.png',dpi=150);plt.show()


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Кривая и единицы — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Настоящий пакетный HF и DFT

В WSL запускаем PySCF: RHF и restricted Kohn–Sham с PBE при одной геометрии и базисе. Функционал и уровень сетки выводим отдельно. Сравниваем собственный RHF с пакетным контрольным значением; равенства HF и DFT не требуем. Это single-point E, без частот, энтропии и свободной энергии активации.

**Перед запуском:** Просканируйте расстояние, затем объясните, почему этот расчёт не даёт ΔG и активационный барьер.

In [4]:
from pyscf import gto, scf, dft
import pyscf
mol=gto.M(atom='H 0 0 0; H 0 0 .74',unit='Angstrom',basis='sto-3g',charge=0,spin=0,verbose=4)
hf=scf.RHF(mol);hf.conv_tol=1e-10;eh=hf.kernel()
ks=dft.RKS(mol);ks.xc='PBE';ks.grids.level=4;ks.conv_tol=1e-10;ed=ks.kernel()
assert hf.converged and ks.converged
assert abs(eh-result['energy_hartree'])<5e-5
artifact={'geometry_angstrom':'H 0 0 0; H 0 0 .74','charge':0,'spin':0,'basis':'sto-3g',
          'pyscf':pyscf.__version__,'functional':'PBE','grid_level':4,
          'RHF_energy_hartree':float(eh),'PBE_energy_hartree':float(ed),
          'HF_converged':bool(hf.converged),'DFT_converged':bool(ks.converged),
          'scope':'single-point electronic energies, not free energies or barriers'}
(OUT/'pyscf_dft.json').write_text(json.dumps(artifact,indent=2))
print(artifact)


System: uname_result(system='Linux', node='miakhlov', release='6.18.58.1-microsoft-standard-WSL2', version='#1 SMP PREEMPT_DYNAMIC Tue Oct 27 00:24:11 UTC 2026', machine='x86_64')  Threads 1
Python 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
numpy 2.5.4  scipy 1.18.1  h5py 3.16.0
Date: Tue Nov 17 14:01:17 2026
PySCF version 2.15.1
PySCF path  /home/miakhlov/course-notebooks/.venv/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 H      0.000000000000   0.000000000000   0.740000000000 AA    0.0000000000

number of shells = 2


number of NR pGTOs = 6


number of NR cGTOs = 2


basis = sto-3g


ecp = {}


... (61 outputs left out)

  HOMO = -0.359710551675669  LUMO = 0.383737151565915  gap/eV = 20.23024


Extra cycle  E= -1.15207279556175  delta_E= 2.22e-16  |g|=    0  |ddm|= 7.11e-16


converged SCF energy = -1.15207279556175


{'geometry_angstrom': 'H 0 0 0; H 0 0 .74', 'charge': 0, 'spin': 0, 'basis': 'sto-3g', 'pyscf': '2.15.1', 'functional': 'PBE', 'grid_level': 4, 'RHF_energy_hartree': -1.1167593073964255, 'PBE_energy_hartree': -1.1520727955617502, 'HF_converged': True, 'DFT_converged': True, 'scope': 'single-point electronic energies, not free energies or barriers'}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Настоящий пакетный HF и DFT — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

RHF/STO-3G при 0,74 Å: −1,116759307±5e−5 Hartree. Пакетные HF/PBE должны сходиться; HF согласуется с собственной реализацией. E не является G или барьером.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':12,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab12/worked/benchmark.json


## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [PySCF: Quickstart](https://pyscf.org/quickstart.html).

Перейдите к [lab12_practice.ipynb](lab12_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.